In [ ]:
import sphere_ref_lib as srl
import sphere_variables as sv

import matplotlib.pyplot as plt

fp = srl.set_output_dir()
fp_nc = srl.set_output_dir_nc()
R, step, xs, d, Z0, R2 = srl.set_basic_params()
R2s, R2, target, apply_ref, _, _ = sv.svv()
e1, e2, H_obs, D_moon, R_moon, e1, e2, tandelta = srl.get_default_param(target)
theta_arr_r2, phi_arr_r2, p0s, p2s, p_hits, ref_dirs, amp_arr, alp_arr = srl.ref_rays_count_ref(R2, xs, d, Z0, R, y=0.0, print_info=True, reflectance=False, e1=e1)

plot_colors = ["red", "darkorange", "springgreen", "mediumblue", "fuchsia"]

In [ ]:
params = {
    "target" : target,
    "R2" : R2,
    "fp_nc" : fp_nc,
    "theta_arr_r2" : theta_arr_r2,
    "phi_arr_r2" : phi_arr_r2,
    "p0s" : p0s,
    "p2s" : p2s,
    "p_hits" : p_hits,
    "ref_dirs" : ref_dirs,
    "amp_arr" : amp_arr,
    "alp_arr" : alp_arr,
}

cors = [
    #("solid_angle", {}),
    ("solid_angle_field", {}),
    ("reflection_rate", {"apply_ref": "average"}),
    ("vertical_direction", {}),
]

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))

ax.grid()

for i in range(len(R2s)):

    R2 = R2s[i]
    ds_all = srl.load_nc_sphere(R2, fp_nc)

    heat_da = srl.make_thph_2dhist(ds_all)

    fin_da = srl.steve_correction_pipeline(heat_da, params, cors)

    phi_min, phi_max = 0.0, 90.0

    fin_da_090 = fin_da.where((fin_da["phi"]>=phi_min) & (fin_da["phi"]<=phi_max), drop=True)
    th_1dcounts_090 = fin_da_090.sum(dim="phi")
    th_1dcounts_090.name = "counts (sum over phi 0-90)"

    
    ax.plot(th_1dcounts_090["theta"].values, th_1dcounts_090.values, c=plot_colors[i], label=f"R2={R2}")
    ax.set_title(f"Counts vs Theta (summed over phi=0-90)")
    ax.set_xlabel("Theta (degrees)")
    ax.set_ylabel("Counts (sum over phi 0-90)")
    #ax.set_ylim(0, 25000)

plt.legend()
plt.show()